In [6]:
# @title #このセルの▶️実行ボタンを押下し、表示されたURLをクリック
import os
import sys
import subprocess
import time
import re
import shutil
import threading
import secrets



_setup_started_at = time.perf_counter()



# 実測値をもとに、URLが表示されるまでの目安を冒頭に簡潔に表示する。
repo_dir = "/content/Irodori-TTS"
_git_dir_for_eta = os.path.join(repo_dir, ".git")
_is_first_setup = not os.path.isdir(_git_dir_for_eta)
try:
    import torch as _torch_for_eta
    _has_gpu = bool(_torch_for_eta.cuda.is_available())
except Exception:
    _has_gpu = False



# 実測：初回約156秒、再実行約23秒。通信やColabの状態による変動分を含めた目安。
if _is_first_setup:
    _eta_min, _eta_max = (120, 240) if _has_gpu else (150, 300)
else:
    _eta_min, _eta_max = (20, 60)



print("AI音声スタジオをセットアップ")
print(f"目安時間：約{_eta_min}〜{_eta_max}秒")



# 1. リポジトリを先に準備する
# 注意: リポジトリ作成前に temp_output を作ると、親フォルダが存在して
# git clone がスキップされる問題があるため、必ず clone を先に実行する。
repo_dir = "/content/Irodori-TTS"
git_dir = os.path.join(repo_dir, ".git")



if not os.path.isdir(git_dir):
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/Aratako/Irodori-TTS.git", repo_dir],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )



os.chdir(repo_dir)



# 2. 必要なディレクトリを準備
os.makedirs("templates", exist_ok=True)
temp_dir = os.path.join(repo_dir, "temp_output")
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir, exist_ok=True)



# 実行ごとに変わるランダムなセキュリティトークンを生成
SECRET_KEY = secrets.token_urlsafe(64)  # 512ビットの暗号学的に安全なランダムキー



# 3. 必要なライブラリをインストール
# Colabで実行中のPythonと同じ環境にインストールする

#print(f"Python: {sys.version.split()[0]}")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "fastapi", "uvicorn", "jinja2", "python-multipart"
    ],
    check=True
)

req_path = os.path.join(repo_dir, "requirements.txt")

if not os.path.isfile(req_path):
    raise FileNotFoundError(
        f"requirements.txt が見つかりません: {req_path}"
    )

# Python 3.13では古いsentencepieceがwheelを利用できず
# ソースビルドに失敗するため、0.2.1以上へ補正する。
with open(req_path, "r", encoding="utf-8") as f:
    requirements = f.read()

if sys.version_info >= (3, 13):
    requirements = re.sub(
        r"(?m)^sentencepiece[^\r\n]*$",
        "sentencepiece>=0.2.1,<0.3",
        requirements
    )

    #print("Python 3.13を検出: sentencepiece>=0.2.1,<0.3 に補正")

fixed_req_path = os.path.join(repo_dir, "requirements_colab.txt")

with open(fixed_req_path, "w", encoding="utf-8") as f:
    f.write(requirements)

#print("Irodori-TTS の依存ライブラリをインストールしています...")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        fixed_req_path
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

if result.returncode != 0:
    print("\n===== pip install エラー =====")
    print(result.stdout)
    print("===== エラーここまで =====\n")

    raise RuntimeError(
        "Irodori-TTS の依存ライブラリのインストールに失敗しました。"
    )

#print("依存ライブラリのインストールが完了しました。")


# 4. パッケージ登録を試みる
# editable install が環境側の都合で失敗しても、app.py 側で repo_dir を
# sys.path に追加するため、インストール失敗を理由に起動全体を止めない。
setup_py_path = os.path.join(repo_dir, "setup.py")
with open(setup_py_path, "w", encoding="utf-8") as f:
    f.write(
        "from setuptools import setup, find_packages\n"
        "setup(name='irodori_tts', version='0.1', packages=find_packages())\n"
    )



try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
except subprocess.CalledProcessError:
    pass



# 5. cloudflared のインストール
if not shutil.which("cloudflared"):
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)



# 4. バックエンド app.py の書き出し
app_code = f"""
import os
import sys
current_dir = os.path.dirname(os.path.abspath(__file__))
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)



import uuid
import shutil
import torch
import time
import threading
import re
import numpy as np
from fastapi import FastAPI, Form, UploadFile, File, Request, HTTPException, Depends
from fastapi.responses import HTMLResponse, FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from irodori_tts.inference_runtime import InferenceRuntime, RuntimeKey, SamplingRequest, save_wav, download_hf_checkpoint
import uvicorn



app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])



SECRET_KEY = "{SECRET_KEY}"



def verify_token(request: Request):
    token = request.query_params.get("key")
    if not token:
        token = request.cookies.get("auth_token")
    if token != SECRET_KEY:
        raise HTTPException(status_code=403, detail="アクセス権限がありません。正しいURLを使用してください。")
    return token



device_name = "CUDA (GPU)" if torch.cuda.is_available() else "CPU"
system_status = {{
    "status": "warming_up",
    "message": f"音声合成モデルをロード中... (デバイス: {{device_name}})",
    "device": "GPU" if torch.cuda.is_available() else "CPU"
}}



runtime = None
checkpoint_id = "Aratako/Irodori-TTS-v4.1-Small"
tasks = {{}}



def load_model_in_memory():
    global runtime, system_status
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model_dir = download_hf_checkpoint(checkpoint_id)
        key = RuntimeKey(checkpoint=str(model_dir), model_device=device, codec_device=device, model_precision="fp32", codec_precision="fp32")
        runtime = InferenceRuntime.from_key(key)
        warmup_req = SamplingRequest(text="システムを起動しました。", ref_wav=None, caption="防音スタジオで録音された高音質でクリアな音声。男性の声。", no_ref=True)
        _ = runtime.synthesize(warmup_req)
        system_status["status"] = "ready"
        system_status["message"] = f"システム準備完了。 ({{device_name}}モード)"
    except Exception as e:
        import traceback
        system_status["status"] = "error"
        system_status["message"] = f"モデル初期化に失敗しました: {{type(e).__name__}}: {{e}}"
        system_status["error"] = traceback.format_exc()
        print("[MODEL_INIT_ERROR]", system_status["error"], flush=True)



threading.Thread(target=load_model_in_memory, daemon=True).start()



@app.get("/")
async def read_index(request: Request, token: str = Depends(verify_token)):
    # トップページだけはURLのkey必須。認証済みCookieがあってもkeyなしURLでは表示しない。
    if request.query_params.get("key") != SECRET_KEY:
        raise HTTPException(status_code=403, detail="アクセス権限がありません。key付きURLを使用してください。")
    with open("templates/index.html", "r", encoding="utf-8") as f:
        html_content = f.read()
    res = HTMLResponse(html_content)
    res.set_cookie(key="auth_token", value=SECRET_KEY, httponly=True)
    return res



@app.get("/status")
async def get_status(token: str = Depends(verify_token)):
    return system_status



def async_generation_worker(task_id, text, mode, voice_description, ref_wav_path, num_steps):
    global runtime
    tasks[task_id] = {{"status": "processing", "progress": 20, "error": None, "file_path": None, "generation_time": 0}}
    start_time = time.time()
    output_path = f"temp_output/{{task_id}}.wav"
    # 原稿中の改行コードを削除し、読み上げ時の不自然な発声を防ぐ。
    processed_text = re.sub(r"[\\r\\n]+", "", str(text or ""))
    try:
        final_description = f"防音スタジオで録音された高音質でクリアな音声。 {{voice_description}}"



        # [br3] や [br0.5] のようなタグを区切りとして扱い、指定秒数（小数可）のデジタル無音を挿入する。
        # タグ自体は読み上げず、文章部分だけを個別に合成して最後に連結する。
        parts = re.split(r'(\\[br\\d+(?:\\.\\d+)?\\])', processed_text, flags=re.IGNORECASE)
        has_pause_tags = bool(re.search(r"\\[br\\d+(?:\\.\\d+)?\\]", processed_text, flags=re.IGNORECASE))
        audio_parts = []
        sample_rate = None
        first_audio_ref_path = f"temp_output/{{task_id}}_first_segment.wav"
        first_audio_saved = False
        text_parts = [p for p in parts if p and not re.fullmatch(r'\\[br\\d+(?:\\.\\d+)?\\]', p, flags=re.IGNORECASE)]
        total_text_parts = max(1, len(text_parts))
        completed_text_parts = 0



        for part in parts:
            if not part:
                continue
            br_match = re.fullmatch(r'\\[br(\\d+(?:\\.\\d+)?)\\]', part, flags=re.IGNORECASE)
            if br_match:
                seconds = float(br_match.group(1))
                if seconds < 0 or seconds > 60:
                    raise ValueError("[br] の無音時間は0〜60秒で指定してください。")
                if audio_parts and sample_rate:
                    last_audio = audio_parts[-1]
                    silence_samples = int(sample_rate * seconds)
                    silence_shape = list(last_audio.shape)
                    silence_shape[-1] = silence_samples
                    if torch.is_tensor(last_audio):
                        silence = torch.zeros(silence_shape, dtype=last_audio.dtype, device=last_audio.device)
                    else:
                        silence = np.zeros(silence_shape, dtype=last_audio.dtype)
                    audio_parts.append(silence)
                continue



            if not part.strip():
                continue
            # [br] タグで分割している場合：
            # ボイスクローン(mode == "clone")は、すべての断片でユーザー指定の参照音声を使い続ける。
            # ボイスデザイン(mode == "design")のみ、2つ目以降を最初の生成音声に寄せる。
            if has_pause_tags and completed_text_parts > 0 and first_audio_saved and mode != "clone":
                req = SamplingRequest(text=("…… " + part), ref_wav=first_audio_ref_path, caption="防音スタジオで録音された高音質でクリアな音声。", no_ref=False, num_steps=num_steps)
            else:
                req = SamplingRequest(text=("…… " + part), ref_wav=ref_wav_path, caption=final_description if mode == "design" else "防音スタジオで録音された高音質でクリアな音声。", no_ref=(mode == "design"), num_steps=num_steps)
            output = runtime.synthesize(req)
            sample_rate = output.sample_rate
            audio_parts.append(output.audio)
            if has_pause_tags and not first_audio_saved:
                save_wav(first_audio_ref_path, output.audio, output.sample_rate)
                first_audio_saved = True
            completed_text_parts += 1
            tasks[task_id]["progress"] = min(90, 40 + int(45 * completed_text_parts / total_text_parts))



        if not audio_parts:
            raise ValueError("音声にする文章を入力してください。")
        if torch.is_tensor(audio_parts[0]):
            combined_audio = torch.cat(audio_parts, dim=-1)
        else:
            combined_audio = np.concatenate([np.asarray(a) for a in audio_parts], axis=-1)
        save_wav(output_path, combined_audio, sample_rate)



        generation_time = round(time.time() - start_time, 2)
        if os.path.exists(output_path):
            tasks[task_id] = {{
                "status": "completed",
                "progress": 100,
                "error": None,
                "file_path": output_path,
                "generation_time": generation_time
            }}
        else:
            tasks[task_id] = {{"status": "failed", "progress": 0, "error": "ファイル生成エラー", "file_path": None, "generation_time": 0}}
    except Exception as e:
        tasks[task_id] = {{"status": "failed", "progress": 0, "error": str(e), "file_path": None, "generation_time": 0}}



@app.post("/generate")
async def request_generate(text: str = Form(...), mode: str = Form("design"), voice_description: str = Form(None), num_steps: int = Form(180), ref_audio: UploadFile = File(None), token: str = Depends(verify_token)):
    if runtime is None:
        return JSONResponse(status_code=503, content={{"error": "エンジンの準備が完了していません。"}})
    if num_steps < 40 or num_steps > 400:
        return JSONResponse(status_code=400, content={{"error": "num_stepsは40〜400の範囲で指定してください。"}})
    task_id = str(uuid.uuid4())
    ref_wav_path = None
    if mode == "clone":
        if not ref_audio or ref_audio.filename == "":
            return JSONResponse(status_code=400, content={{"error": "ファイルを選択してください。"}})
        ref_path = f"temp_output/{{task_id}}_ref{{os.path.splitext(ref_audio.filename)[1]}}"
        with open(ref_path, "wb") as buffer:
            shutil.copyfileobj(ref_audio.file, buffer)
        ref_wav_path = ref_path



    threading.Thread(target=async_generation_worker, args=(task_id, text, mode, voice_description, ref_wav_path, num_steps), daemon=True).start()
    return {{"task_id": task_id}}



@app.get("/task/{{task_id}}")
async def check_task_status(task_id: str, token: str = Depends(verify_token)):
    if task_id not in tasks:
        return JSONResponse(status_code=404, content={{"error": "不明なタスクです。"}})
    return tasks[task_id]



@app.get("/download/{{task_id}}")
async def download_file(task_id: str, token: str = Depends(verify_token)):
    if task_id not in tasks or tasks[task_id]["status"] != "completed":
        return JSONResponse(status_code=404, content={{"error": "ダウンロード不可"}})
    return FileResponse(tasks[task_id]["file_path"], media_type="audio/wav", filename="generated.wav")



if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=7861)
"""



with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip())



# 5. HTMLテンプレートの書き出し
html_path = "templates/index.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<!DOCTYPE html>\n<html lang=\"ja\">\n<head>\n    <meta charset=\"UTF-8\">\n    <title>アデランス - AI音声スタジオ</title>\n    <link href=\"https://cdnjs.cloudflare.com/ajax/libs/tailwindcss/2.2.19/tailwind.min.css\" rel=\"stylesheet\">\n    <style>\n")
    f.write("        body { background-color: #faf9f6; color: #2d3139; font-family: sans-serif; }\n")
    f.write("        .brand-header { background: #1a1a1a; border-bottom: 2px solid #dfc48c; }\n")
    f.write("        .gold-accent-text { color: #dfc48c; }\n")
    f.write("        .preset-card { cursor: pointer; border: 1px solid #e2e4e8; background-color: #ffffff; border-radius: 12px; padding: 12px; transition: all 0.2s; height: 105px; display: flex; flex-direction: column; justify-content: flex-start; overflow: hidden; }\n")
    f.write("        .preset-card:hover { border-color: #dfc48c; box-shadow: 0 4px 12px rgba(223,196,140,0.15); }\n")
    f.write("        .preset-card.active { border-color: #bfa05d; background-color: #fdfcfa; box-shadow: 0 0 0 2px rgba(191,160,93,0.15); }\n")
    f.write("        .premium-btn { background: linear-gradient(135deg, #d2b474, #bfa05d); color: #ffffff; font-weight: bold; }\n")
    f.write("        .premium-download-btn { background: #4a3e3d; color: #ffffff; font-weight: bold; }\n")
    f.write("        .premium-download-btn:hover { background: #dfc48c; color: #4a3e3d; }\n")
    f.write("        .luxury-card { background-color: #ffffff; border: 1px solid #edebe6; }\n")
    f.write("        .badge { font-size: 10px; font-weight: bold; padding: 2px 8px; border-radius: 4px; }\n")
    f.write("        .badge-male { background-color: #e0f2fe; color: #0369a1; }\n")
    f.write("        .badge-female { background-color: #fce7f3; color: #be185d; }\n")
    f.write("        .badge-neutral { background-color: #f3e8ff; color: #6b21a8; }\n")
    f.write("        .badge-other { background-color: #f3f4f6; color: #4b5563; }\n")
    f.write("        .filter-tab { padding: 6px 12px; font-size: 12px; border-radius: 6px; border: 1px solid #e5e7eb; background: #fff; cursor: pointer; }\n")
    f.write("        .filter-tab.active { background: #1f2937; color: #fff; }\n")
    f.write("        #loadingOverlay { background: radial-gradient(circle at center, #181715 0%, #080807 100%); transition: opacity 1s cubic-bezier(0.4, 0, 0.2, 1); }\n")
    f.write("        .glowing-aura { position: relative; display: flex; align-items: center; justify-content: center; width: 140px; height: 140px; }\n")
    f.write("        .aura-wave { position: absolute; border: 2px solid rgba(223,196,140,0.35); border-radius: 50%; width: 100%; height: 100%; animation: auraPulse 2.2s cubic-bezier(0.215, 0.61, 0.355, 1) infinite; }\n")
    f.write("        .aura-wave:nth-child(2) { animation-delay: 0.7s; }\n")
    f.write("        .aura-wave:nth-child(3) { animation-delay: 1.4s; }\n")
    f.write("        @keyframes auraPulse { 0% { transform: scale(0.6); opacity: 1; } 100% { transform: scale(1.6); opacity: 0; box-shadow: 0 0 40px rgba(223,196,140,0.3); } }\n")
    f.write("        .mic-core { z-index: 10; background: radial-gradient(circle, #2d2922 0%, #151310 100%); border: 2px solid #dfc48c; box-shadow: 0 0 30px rgba(223,196,140,0.35); border-radius: 50%; width: 84px; height: 84px; display: flex; align-items: center; justify-content: center; }\n")
    f.write("    </style>\n</head>\n<body class=\"min-h-screen flex flex-col justify-between\">\n")



    f.write("    <div id=\"loadingOverlay\" class=\"fixed inset-0 z-50 flex flex-col items-center justify-center text-center bg-black\">\n")
    f.write("        <div class=\"glowing-aura mb-8\">\n")
    f.write("            <div class=\"aura-wave\"></div>\n")
    f.write("            <div class=\"aura-wave\"></div>\n")
    f.write("            <div class=\"aura-wave\"></div>\n")
    f.write("            <div class=\"mic-core\">\n")
    f.write("                <svg class=\"w-10 h-10\" fill=\"none\" stroke=\"#dfc48c\" stroke-width=\"1.5\" viewBox=\"0 0 24 24\" xmlns=\"http://www.w3.org/2000/svg\">\n")
    f.write("                    <rect x=\"9\" y=\"2\" width=\"6\" height=\"10\" rx=\"3\" fill=\"#2d2922\" stroke=\"#dfc48c\" stroke-width=\"1.5\"/>\n")
    f.write("                    <path d=\"M5 10c0 3.866 3.134 7 7 7s7-3.134 7-7M12 17v4m-4 0h8\" stroke-linecap=\"round\" stroke-linejoin=\"round\"/>\n")
    f.write("                </svg>\n")
    f.write("            </div>\n")
    f.write("        </div>\n")
    f.write("        <h2 class=\"text-2xl font-bold text-white mb-2 tracking-widest\">ADERANS <span class=\"gold-accent-text\">AI VOICE STUDIO</span></h2>\n")
    f.write("        <p id=\"statusText\" class=\"text-sm text-gray-400 font-medium tracking-wide mb-2\">システムを準備中です...</p>\n")
    f.write("        <p id=\"initTimerText\" class=\"text-xs font-mono font-bold text-yellow-500 mb-6\">ロード中（経過時間：0秒／目安：約120秒）</p>\n")
    f.write("        <div class=\"w-64 bg-gray-800 rounded-full h-2 overflow-hidden mx-auto border border-gray-700\">\n")
    f.write("            <div id=\"initProgressBar\" class=\"bg-yellow-500 h-2 rounded-full transition-all duration-700\" style=\"width: 8%\"></div>\n")
    f.write("        </div>\n")
    f.write("    </div>\n")



    f.write("    <header class=\"brand-header py-4\">\n")
    f.write("        <div class=\"container mx-auto px-6 max-w-6xl flex justify-between items-center\">\n")
    f.write("            <h1 class=\"text-lg font-bold text-white tracking-wider\">ADERANS <span class=\"gold-accent-text\">AI VOICE STUDIO</span></h1>\n")
    f.write("            <span id=\"headerDeviceBadge\"></span>\n")
    f.write("        </div>\n")
    f.write("    </header>\n")
    f.write("    <main class=\"container mx-auto px-4 py-8 flex-grow max-w-6xl\">\n")
    f.write("        <div id=\"cpuWarningBanner\" class=\"hidden mb-6 bg-yellow-50 border border-yellow-200 rounded-xl p-4 flex items-start space-x-3 text-sm text-yellow-800\">\n")
    f.write("            <span class=\"text-lg\">⚠️</span>\n")
    f.write("            <div>\n")
    f.write("                <p class=\"font-bold\">CPU推論モードで動作中</p>\n")
    f.write("                <p class=\"text-xs mt-0.5\">現在GPUを使用できないため、音声の生成には時間がかかります（長文の場合は数分程度かかります）。</p>\n")
    f.write("            </div>\n")
    f.write("        </div>\n")
    f.write("        <div class=\"grid grid-cols-1 lg:grid-cols-12 gap-8\">\n")
    f.write("            <div class=\"lg:col-span-6 flex flex-col\">\n")
    f.write("                <div class=\"luxury-card rounded-2xl p-6 shadow-sm flex-grow flex flex-col\" style=\"height: 600px;\">\n")
    f.write("                    <div class=\"mb-4\">\n")
    f.write("                        <h3 class=\"text-sm font-bold text-gray-800 tracking-wider mb-3\">🎙️ ボイスプリセット(50 パターン)</h3>\n")
    f.write("                        <div class=\"flex flex-wrap gap-1.5\">\n")
    f.write("                            <button id=\"tab-all\" class=\"filter-tab active\" onclick=\"filterCat('all')\">すべて</button>\n")
    f.write("                            <button id=\"tab-male\" class=\"filter-tab\" onclick=\"filterCat('male')\">男性</button>\n")
    f.write("                            <button id=\"tab-female\" class=\"filter-tab\" onclick=\"filterCat('female')\">女性</button>\n")
    f.write("                            <button id=\"tab-neutral\" class=\"filter-tab\" onclick=\"filterCat('neutral')\">中性</button>\n")
    f.write("                            <button id=\"tab-other\" class=\"filter-tab\" onclick=\"filterCat('other')\">その他</button>\n")
    f.write("                        </div>\n")
    f.write("                    </div>\n")
    f.write("                    <div class=\"flex-grow overflow-y-auto space-y-2.5 pr-1\" id=\"presetList\"></div>\n")
    f.write("                </div>\n")
    f.write("            </div>\n")
    f.write("            <div class=\"lg:col-span-6 space-y-6\">\n")
    f.write("                <div class=\"luxury-card rounded-2xl p-6 shadow-sm\">\n")
    f.write("                    <div class=\"flex bg-gray-100 p-1.5 rounded-xl mb-6\">\n")
    f.write("                        <button id=\"tabDesign\" class=\"tab-btn flex-1 py-2.5 text-center text-sm font-bold text-gray-800 rounded-lg bg-white shadow-sm border border-gray-200\" onclick=\"switchMode('design')\">🎭 ボイスデザイン</button>\n")
    f.write("                        <button id=\"tabClone\" class=\"tab-btn flex-1 py-2.5 text-center text-sm font-bold text-gray-400 rounded-lg\" onclick=\"switchMode('clone')\">🎙️ ボイスクローン</button>\n")
    f.write("                    </div>\n")
    f.write("                    <form id=\"ttsForm\" class=\"space-y-5\">\n")
    f.write("                        <input type=\"hidden\" id=\"modeInput\" name=\"mode\" value=\"design\">\n")
    f.write("                        <div>\n")
    f.write("                            <p class=\"text-xs text-gray-500 mb-2\">💡 [br1] は1秒、[br1.5] は1.5秒、[br0.8] は0.8秒の無音を機械的に挿入</p>\n")
    f.write("                            <div class=\"flex justify-between items-center mb-2\">\n")
    f.write("                                <label class=\"block text-xs font-bold text-gray-500\">🔊 生成原稿テキスト</label>\n")
    f.write("                                <span id=\"charCounter\" class=\"text-xs font-bold text-gray-400\">処理目安時間: 計算中</span>\n")
    f.write("                            </div>\n")
    f.write("                            <textarea id=\"ttsText\" name=\"text\" rows=\"5\" class=\"w-full p-4 bg-gray-50 border border-gray-200 rounded-xl text-sm text-gray-900 focus:outline-none\" placeholder=\"喋らせる文章を入力してください。[br1.5]や[br2]で無音を挿入できます。\" required oninput=\"updateEstimation()\"></textarea>\n")
    f.write("                        </div>\n")
    f.write("                        <div id=\"designArea\">\n")
    f.write("                            <label class=\"block text-xs font-bold text-gray-500 mb-2\">🎭 キャラクター記述</label>\n")
    f.write("                            <input type=\"text\" id=\"voiceDescription\" name=\"voice_description\" class=\"w-full p-4 bg-gray-50 border border-gray-200 rounded-xl text-sm text-gray-900 focus:outline-none\" placeholder=\"例：落ち着いた上品な大人の声。\">\n")
    f.write("                        </div>\n")
    f.write("                        <div class=\"bg-amber-50 border border-amber-200 rounded-xl p-4\">\n")
    f.write("                            <label for=\"numSteps\" class=\"block text-xs font-bold text-gray-700 mb-2\">⚙️ 音声品質（サンプリングステップ数）</label>\n")
    f.write("                            <select id=\"numSteps\" name=\"num_steps\" class=\"w-full p-3 bg-white border border-gray-200 rounded-lg text-sm\">\n")
    f.write("                                <option value=\"40\">40（速い・品質低下・CPU時推奨）</option>\n")
    f.write("                                <option value=\"60\">60</option>\n")
    f.write("                                <option value=\"80\">80</option>\n")
    f.write("                                <option value=\"100\">100</option>\n")
    f.write("                                <option value=\"120\">120</option>\n")
    f.write("                                <option value=\"140\">140</option>\n")
    f.write("                                <option value=\"160\">160</option>\n")
    f.write("                                <option value=\"180\">180</option>\n")
    f.write("                                <option value=\"200\" selected>200（標準・バランス）</option>\n")
    f.write("                                <option value=\"220\">220</option>\n")
    f.write("                                <option value=\"240\">240</option>\n")
    f.write("                                <option value=\"260\">260</option>\n")
    f.write("                                <option value=\"280\">280</option>\n")
    f.write("                                <option value=\"300\">300</option>\n")
    f.write("                                <option value=\"320\">320</option>\n")
    f.write("                                <option value=\"340\">340</option>\n")
    f.write("                                <option value=\"360\">360</option>\n")
    f.write("                                <option value=\"380\">380</option>\n")
    f.write("                                <option value=\"400\">400（遅い・高品質）</option>\n")
    f.write("                            </select>\n")
    f.write("                            <p class=\"text-xs text-gray-500 mt-2\">値を上げると生成時間が長くなります。品質向上は文章やモデルによって異なります。</p>\n")
    f.write("                            <script>document.addEventListener('DOMContentLoaded', () => { const stepsEl = document.getElementById('numSteps'); if (stepsEl) stepsEl.addEventListener('change', updateEstimation); });</script>\n")
    f.write("                        </div>\n")
    f.write("                        <div id=\"cloneArea\" class=\"hidden\">\n")
    f.write("                            <label class=\"block text-xs font-bold text-gray-500 mb-2\">🎙️ 複製ターゲットの音声 (.wav)</label>\n")
    f.write("                            <input type=\"file\" id=\"refAudioInput\" name=\"ref_audio\" class=\"w-full p-3 border border-dashed border-gray-300 rounded-xl text-sm bg-gray-50 text-gray-500\">\n")
    f.write("                        </div>\n")
    f.write("                        <button type=\"submit\" id=\"generateBtn\" class=\"w-full premium-btn py-4 rounded-xl shadow-sm font-bold text-sm tracking-widest\">音声を生成する</button>\n")
    f.write("                    </form>\n")
    f.write("                </div>\n")



    f.write("                <div id=\"progressArea\" class=\"hidden bg-white border border-yellow-200 shadow-sm rounded-2xl p-6 space-y-4\">\n")
    f.write("                    <div class=\"flex justify-between items-center\">\n")
    f.write("                        <h4 id=\"progressStatus\" class=\"font-bold text-gray-800 text-sm\">⏳ 音声を生成中...</h4>\n")
    f.write("                        <span id=\"liveTimer\" class=\"text-xs font-mono font-bold text-yellow-800 bg-yellow-50 border border-yellow-200 px-2.5 py-1 rounded-lg\">残り約 -- 秒</span>\n")
    f.write("                    </div>\n")
    f.write("                    <div class=\"w-full bg-gray-100 rounded-full h-3.5 overflow-hidden p-0.5 border border-gray-200\">\n")
    f.write("                        <div id=\"progressBar\" class=\"bg-gradient-to-r from-yellow-600 to-yellow-500 h-2.5 rounded-full transition-all duration-300\" style=\"width: 5%\"></div>\n")
    f.write("                    </div>\n")
    f.write("                    <p id=\"progressDetailText\" class=\"text-[11px] text-gray-500 text-center\">AIが音声を合成しています（目安: 約 -- 秒）</p>\n")
    f.write("                </div>\n")



    f.write("                <div id=\"resultArea\" class=\"hidden bg-gray-50 border border-gray-200 rounded-2xl p-6 space-y-4\">\n")
    f.write("                    <h4 class=\"font-bold text-gray-800 text-sm\">🎉 音声が完成しました</h4>\n")
    f.write("                    <audio id=\"audioPlayer\" controls class=\"w-full\"></audio>\n")
    f.write("                    <a id=\"downloadBtn\" href=\"#\" download=\"generated.wav\" class=\"block text-center premium-download-btn font-bold py-4 rounded-xl shadow-sm text-sm\">📁 ファイルをダウンロード</a>\n")
    f.write("                </div>\n")
    f.write("            </div>\n")
    f.write("        </div>\n")
    f.write("    </main>\n")
    f.write("    <footer class=\"bg-white border-t border-gray-200 text-center py-4 text-xs text-gray-400\">© ADERANS - AI VOICE STUDIO.</footer>\n")



    f.write("    <script>\n")
    f.write("        const presets = [\n")
    f.write("            { id: 'p0', cat: 'female', badge: '女性', title: '📞 電話応対トレーナー（🔥）', desc: '女性の声。 落ち着きと丁寧さがあり、実践的な電話応対のポイントをわかりやすく伝える研修ナレーション。', text: 'お電話を受けた時点で、お客様が詳しく状況をご説明くださる場合があります。突然詳細なお話をされると、メモを取る時間が十分に確保できないこともあります。[br1]そのような場合は、「何点かお伺いしてもよろしいでしょうか。」ではなく、「お伺いした内容を確認させていただいてもよろしいでしょうか。」とお伝えし、お客様から伺った内容を整理しながら確認しましょう。[br1.5]内容を復唱して確認することで、聞き漏れや認識の相違を防ぐことができます。' },\n")
    f.write("            { id: 'p1', cat: 'female', badge: '女性', title: '💼 ビジネスナレーター', desc: '女性の声。 上品で知的な大人の女性。ビジネスシーンや各種ナレーション。', text: '本日はアデランス AI音声スタジオへお越しいただきありがとうございます。高品質クオリティで設計されております。' },\n")
    f.write("            { id: 'p2', cat: 'male', badge: '男性', title: '🏴\u200d☠️ 豪快な海賊（🔥）', desc: '男性の声。極めて低く、重厚で野太いバスボイス。喉の奥から絞り出すような、濁りと強いザラつきのある荒々しいダミ声。酒焼けしたようなしゃがれと、獣のような低い唸りを含む。若々しい爽やかさや、滑らかで綺麗な発声は一切なく、粗野で威圧的な大男の声。ゆっくりと腹の底から響かせ、言葉の一つ一つに重量感を持たせる。豪快で不気味な笑い声が似合う、恐れられる伝説の大海賊。', text: 'おやおや……こいつぁ、面白いことになってきたじゃねぇか。ヒッヒッヒ……。さぁ、野郎ども！　帆を上げろ！　風はこっちの味方だ。お宝は目の前……逃すんじゃねぇぞ！' },\n")
    f.write("            { id: 'p3', cat: 'female', badge: '女性', title: '🏨 ホテルコンシェルジュ', desc: '女性の声。 丁寧な話し方。暖かみがあり、抜群の信頼感を与えてくれる声。', text: 'お客様、本日のご予定はいかがでしょうか。ご不明な点がございましたら、いつでもお気軽にお尋ねください。' },\n")
    f.write("            { id: 'p4', cat: 'male', badge: '男性', title: '💡 IT製品解説', desc: '男性の声。 明瞭な発音と自信に満ちたモダンなピッチ感。製品紹介向け。', text: 'ご紹介するのは次世代の音響処理技術です。余計なノイズを遮断し、極限のサウンド体験を瞬時に提供します。' },\n")
    f.write("            { id: 'p5', cat: 'female', badge: '女性', title: '📈 セミナー専任講師', desc: '女性の声。 クリアではきはきとした聴き取りやすいスピードの教員風ボイス。', text: 'それでは本日のセッションを開始します。今回の重要なポイントは、効率的なアプローチにあります。' },\n")
    f.write("            { id: 'p6', cat: 'male', badge: '男性', title: '👔 新卒採用会社説明', desc: '男性の声。 フレッシュで熱意の伝わる、ハツラツとした爽やかな男性。', text: '当社のブースへようこそ！私たちは新しい未来を共に切り拓く、パッション溢れる仲間を求めています。' },\n")
    f.write("            { id: 'p7', cat: 'male', badge: '男性', title: '🎙️ 深みのある渋い重低音', desc: '男性の声。 低音で渋みのある男性。ゆったりとした安心感のあるトーン。', text: 'お疲れ様です。今日も一日本当によく頑張りましたね。コーヒーでも飲みながら、安らぎの時間をお過ごしください。' },\n")
    f.write("            { id: 'p8', cat: 'male', badge: '男性', title: '👨 優しく包み込むお父さん', desc: '男性の声。 温厚で、包容力のある安心できる中高年の声。', text: 'よく帰ってきたね。温かいご飯ができているから、まずはゆっくり休みなさい。焦らなくても大丈夫だ。' },\n")
    f.write("            { id: 'p9', cat: 'male', badge: '男性', title: '📣 スポーツ実況DJ', desc: '男性の声。 テンションが高く、エネルギッシュでダイナミックなボイス。', text: 'さあ、いよいよ決戦の瞬間がやってまいりました！スタジアムの興奮は最高潮、歴史に残る一戦が幕を開けます！' },\n")
    f.write("            { id: 'p10', cat: 'male', badge: '男性', title: '🐺 クールな青年キャラ', desc: '男性の声。 低めのトーンで、スマートかつ少し影のあるキャラクター。', text: 'フッ…、どうやら少し手間取っているようだな。俺に任せておけ。期待以上の結果は出してみせるさ。' },\n")
    f.write("            { id: 'p11', cat: 'male', badge: '男性', title: '🍂 癒やしのささやき朗読', desc: '男性の声。 非常に優しい、聴く人を穏やかに眠りに誘うような甘い声。', text: '目を閉じて、静かな呼吸を繰り返してください。今日の疲れをすべて夜風に預けて、ゆっくりと眠りましょう。' },\n")
    f.write("            { id: 'p12', cat: 'male', badge: '男性', title: '🛹 快活なストリート青年', desc: '男性の声。 明るく親しみやすい、人懐っこいヤングトーン。', text: 'みんな、調子はどう？今日の風は最高に気持ちいいね！細かいことは気にせず、思いっきり楽しんでいこうぜ！' },\n")
    f.write("            { id: 'p13', cat: 'female', badge: '女性', title: '📻 FMラジオDJ', desc: '女性の声。 滑らかな滑舌と、深夜の落ち着いたラジオを彷彿とさせる声。', text: '時刻は間もなく午前零時を迎えます。今日を締めくくる最後のひととき、心地よいメロディーとお過ごしください。' },\n")
    f.write("            { id: 'p21', cat: 'neutral', badge: '中性', title: '🧙 謎めいた賢者魔法使い', desc: '中性的な声。 年齢不詳の、知恵を蓄えた深みのある中性的なボイス。', text: 'ほほう、ここまでたどり着くとはな。お主が求める真理への扉は、すでに心の中に開かれているのやもしれんぞ…。' },\n")
    f.write("            { id: 'p22', cat: 'neutral', badge: '中性', title: '🧛 ミステリアス中性ダーク', desc: '中性的な声。 色気があり、鋭さと静けさを兼ね備えたダークな中性声音。', text: '正義、か…。その甘い幻想が、いつまで通用するか試してやろう。闇に沈む前に、抗ってみせるがいい。' },\n")
    f.write("            { id: 'p23', cat: 'neutral', badge: '中性', title: '🔮 占い師 of リーディング', desc: '中性的な声。 ミステリアスで、運命を解き明かす神秘的なウィスパー調中性ボイス。', text: 'カードは語っています。あなたの目の前にある迷いは、間もなくまばゆい光に照らされ、解決の糸口を見出すでしょう。' },\n")
    f.write("            { id: 'p24', cat: 'unspecified', badge: '指定なし', title: '🚀 SFアンドロイド案内（🔥）', desc: '性別を特定しない声。 感情を排した無機質かつ先進的なコックピットアナウンス。', text: '警告。システムへの不正アクセスを検知しました。プロトコルに基づき、速やかに防衛シールドを構築します。' },\n")
    f.write("            { id: 'p25', cat: 'female', badge: '女性', title: '📢 明るいPR・セールス', desc: '女性の声。 ハキハキとした張りのある声。特売や広告、お買い得情報向け。', text: '皆様にうれしいお知らせです！本日より期間限定の特別なキャンペーンがスタートいたします。どうぞお見逃しなく！' },\n")
    f.write("            { id: 'p26', cat: 'female', badge: '女性', title: '🏥 医療機関ガイダンス', desc: '女性の声。 落ち着いて一歩一歩伝える、穏やかで安心感のある窓口トーン。', text: '本日はご来院ありがとうございます。問診票のご記入がお済みの方は、受付の緑色のトレイへお入れになりお待ちください。' },\n")
    f.write("            { id: 'p27', cat: 'female', badge: '女性', title: '⚠️ 災害・緊急防災放送', desc: '女性の声。 極めて聞き取りやすく、正確性を備えたアナウンス。', text: 'システムより防災情報をお伝えいたします。ただいま機器の自動調整が正常に完了いたしました。' },\n")
    f.write("            { id: 'p28', cat: 'male', badge: '男性', title: '🏫 美術館音声ガイド', desc: '男性の声。 教養の感じられる、ゆっくりとした優雅な芸術解説ボイス。', text: '皆様、右手に見えますのが代表的な傑作です。画家の筆使いが、光の陰影をドラマチックに描き出しています。' },\n")
    f.write("            { id: 'p29', cat: 'female', badge: '女性', title: '🚌 観光バス案内ガイド', desc: '女性の声。 情緒的で親しみのある、旅行を盛り上げるガイドさんボイス。', text: '皆様、左手に見えてまいりましたのは名所でございます。春には美しい桜が一面に咲き乱れるスポットです。' },\n")
    f.write("            { id: 'p30', cat: 'unspecified', badge: '指定なし', title: '🤖 自動音声応答(IVR)', desc: '性別を特定しない声。 コールセンターやカスタマーサポート用の標準自動応答。', text: 'お電話ありがとうございます。ナビダイヤルでお繋ぎします。オペレーターへのお問い合わせは「いち」を押してください。' },\n")
    f.write("            { id: 'p31', cat: 'female', badge: '女性', title: '🌸 透明感のあるヒロイン', desc: '女性の声。 明るく澄んだ声。アニメの主人公や爽やかな作品紹介に。', text: '新しい朝が始まるよ。昨日までの不安も、今日の一歩で変えていこう！' },\n")
    f.write("            { id: 'p32', cat: 'female', badge: '女性', title: '🎀 元気いっぱいの後輩', desc: '女性の声。 弾むように明るく、親しみやすい若々しい声。', text: '先輩、準備できましたか？今日は思いっきり楽しんでいきましょう！' },\n")
    f.write("            { id: 'p33', cat: 'female', badge: '女性', title: '🧊 クールなプロフェッショナル', desc: '女性の声。 感情を抑えた端正な声。知的な解説やクールなキャラクター向け。', text: '状況を整理しました。必要な情報はすべて揃っています。次の手順へ進みましょう。' },\n")
    f.write("            { id: 'p34', cat: 'female', badge: '女性', title: '🧚 妖精のささやき', desc: '女性の声。 軽やかで幻想的な雰囲気。ファンタジー作品やゲームに。', text: '森の奥で、小さな光があなたを待っています。静かについてきてくださいね。' },\n")
    f.write("            { id: 'p35', cat: 'female', badge: '女性', title: '👑 気品ある女王', desc: '女性の声。 堂々として優雅。威厳のあるセリフや物語の語りに。', text: 'よくぞここまで参りました。あなたの勇気に、心から敬意を表しましょう。' },\n")
    f.write("            { id: 'p36', cat: 'female', badge: '女性', title: '🕵️ ミステリー探偵', desc: '女性の声。 落ち着いた観察眼を感じさせる、含みのある声。', text: '証拠は嘘をつきません。見落とした小さな違和感こそ、真実への鍵です。' },\n")
    f.write("            { id: 'p37', cat: 'female', badge: '女性', title: '🍵 ほっとする癒やし声', desc: '女性の声。 柔らかく穏やか。日常系コンテンツやリラックス音声向け。', text: '今日もお疲れさまでした。温かいお茶でも飲んで、ゆっくりひと息つきましょう。' },\n")
    f.write("            { id: 'p38', cat: 'female', badge: '女性', title: '🎤 アイドルMC', desc: '女性の声。 華やかで弾けるようなテンション。イベントやライブの進行に。', text: 'みんなー！今日は来てくれてありがとう！最後まで一緒に盛り上がっていこうね！' },\n")
    f.write("            { id: 'p39', cat: 'female', badge: '女性', title: '🧙\u200d♀️ 闇の魔女', desc: '女性の声。 妖しく落ち着いた声。ダークファンタジーの演出向け。', text: 'ふふふ、扉は開かれたわ。あなたが何を選ぶのか、じっくり見せてもらいましょう。' },\n")
    f.write("            { id: 'p40', cat: 'female', badge: '女性', title: '📰 ニュースキャスター', desc: '女性の声。 明瞭で安定感のある報道調。ニュースや情報読み上げに。', text: '続いての話題です。新しい取り組みが発表され、今後の展開に注目が集まっています。' },\n")
    f.write("            { id: 'p41', cat: 'female', badge: '女性', title: '🧸 幼い子どもの声', desc: '女性の声。 無邪気で素直な印象。かわいらしいキャラクター向け。', text: 'ねえねえ、あっちにキラキラしたものがあるよ！一緒に見に行こう！' },\n")
    f.write("            { id: 'p42', cat: 'female', badge: '女性', title: '🌙 大人のアンニュイ', desc: '女性の声。 少し気だるく、落ち着いた色気のあるトーン。', text: '急がなくてもいいんじゃない？夜はまだ長いし、ゆっくり話しましょう。' },\n")
    f.write("            { id: 'p43', cat: 'male', badge: '男性', title: '🦸 正統派ヒーロー', desc: '男性の声。 力強くまっすぐな声。決意表明やアクション作品向け。', text: 'ここで諦めるわけにはいかない。みんなの未来は、俺たちが守る！' },\n")
    f.write("            { id: 'p44', cat: 'male', badge: '男性', title: '🎩 紳士のナレーション', desc: '男性の声。 丁寧で余裕のある大人の声。上質な紹介や朗読に。', text: 'どうぞおくつろぎください。本日は、特別なひとときをご案内いたします。' },\n")
    f.write("            { id: 'p45', cat: 'male', badge: '男性', title: '🧑\u200d🍳 明るい料理人', desc: '男性の声。 活気があり親しみやすい声。料理動画やグルメ紹介に。', text: 'さあ、ここからが腕の見せどころ！ひと手間加えるだけで、味がぐっと変わります。' },\n")
    f.write("            { id: 'p46', cat: 'male', badge: '男性', title: '🧑\u200d🚀 宇宙飛行士', desc: '男性の声。 冷静さの中に好奇心がある声。SFや科学解説向け。', text: 'こちら管制室、準備完了。窓の向こうには、まだ誰も見たことのない景色が広がっています。' },\n")
    f.write("            { id: 'p47', cat: 'male', badge: '男性', title: '🧟 不気味な怪人', desc: '男性の声。 低く不穏な声。ホラーや敵キャラクターの演出に。', text: '逃げても無駄だ……この場所に足を踏み入れた時から、物語は始まっている。' },\n")
    f.write("            { id: 'p48', cat: 'male', badge: '男性', title: '📊 エグゼクティブプレゼン', desc: '男性の声。 説得力のある中音域。言葉一つ一つに重みを持たせた信頼のボイス。', text: '我々が目指すビジョンは単なる技術の革新に留まりません。ライフスタイルそのものを豊かにすることにあります。' },\n")
    f.write("            { id: 'p49', cat: 'male', badge: '男性', title: '📚 知的な解説者', desc: '男性の声。 論理的で聞き取りやすい声。解説動画や教材に。', text: '大切なのは、結果だけを見ることではありません。そこに至る過程を理解することです。' },\n")
    f.write("            { id: 'p50', cat: 'male', badge: '男性', title: '🎮 ゲーム実況者', desc: '男性の声。 リアクション豊かでテンポのよい声。ゲーム配信や紹介向け。', text: 'おおっ、これはすごい！まさかここで隠しイベントが来るとは思わなかった！' },\n")
    f.write("            { id: 'p51', cat: 'male', badge: '男性', title: '🧓 人情味のある親方', desc: '男性の声。 豪快さと温かみを感じる年長者の声。', text: '失敗したっていいんだ。大事なのは、そこから何を学んで次にどう動くかだ。' },\n")
    f.write("            { id: 'p52', cat: 'neutral', badge: '中性', title: '🪄 夢見る語り部', desc: '中性的な声。 柔らかく性別を感じさせにくい幻想的な声。', text: '星がひとつ瞬くたび、遠い世界で新しい物語が生まれているのです。' },\n")
    f.write("            { id: 'p53', cat: 'neutral', badge: '中性', title: '🖥️ AIオペレーター', desc: '中性的な声。 正確で無機質。システム案内やSFの音声UI向け。', text: '接続を確認しました。データの同期が完了しました。次の操作を入力してください。' },\n")
    f.write("            { id: 'p54', cat: 'neutral', badge: '中性', title: '🎭 舞台の語り手', desc: '中性的な声。 表情豊かで物語を引き込む朗読調。', text: '幕が上がるその瞬間、誰も知らなかった運命が静かに動き始めた。' },\n")
    f.write("            { id: 'p55', cat: 'neutral', badge: '中性', title: '🌀 夢幻の存在', desc: '中性的な声。 浮遊感があり、現実離れした不思議な印象。', text: 'ここは夢と現実のあわい。あなたの想像したものが、少しずつ形になっていきます。' },\n")
    f.write("            { id: 'p56', cat: 'unspecified', badge: '指定なし', title: '🔔 駅構内アナウンス', desc: '性別を特定しない声。 簡潔で明瞭な案内放送。施設や交通機関の案内向け。', text: 'ご利用ありがとうございます。まもなく列車が到着します。黄色い線の内側でお待ちください。' },\n")
    f.write("            { id: 'p57', cat: 'unspecified', badge: '指定なし', title: '🎬 映画予告ナレーター（🔥）', desc: '性別を特定しない声。 期待感を高める重厚な語り。予告や紹介映像に。', text: 'その出会いが、世界を変える。誰も予想できなかった物語が、いま始まる。' },\n")
    f.write("        ];\n")
    f.write("        let activeCategory = 'all'; let isCpuMode = false;\n")
    f.write("        let secPerChar = null;\n")
    f.write("        function getEstimatedDuration() {\n")
    f.write("            const text = document.getElementById('ttsText').value || '';\n")
    f.write("            const pauseMatches = [...text.matchAll(/\\[br(\\d+(?:\\.\\d+)?)\\]/gi)];\n")
    f.write("            const pauseSeconds = pauseMatches.reduce((sum, match) => sum + Number(match[1]), 0);\n")
    f.write("            const spokenText = text.replace(/\\[br(\\d+(?:\\.\\d+)?)\\]/gi, '');\n")
    f.write("            const n = Math.max(1, spokenText.length);\n")
    f.write("            const stepsEl = document.getElementById('numSteps');\n")
    f.write("            const steps = stepsEl ? Number(stepsEl.value || 180) : 100;\n")
    f.write("            const stepMultiplier = steps / 100; // 100ステップを基準に目安時間を補正\n")
    f.write("            const fallbackRateAt100 = isCpuMode ? 0.45 : 0.018;\n")
    f.write("            const rateAt100 = secPerChar == null ? fallbackRateAt100 : secPerChar;\n")
    f.write("            const rawEstimate = Math.round(1.5 + n * rateAt100 * stepMultiplier);\n")
    f.write("            // 実測補正：以前の係数7.66に、今回の目安31秒・実績50秒（約1.613倍）を反映。初回係数は約12.35。\n")
    f.write("            // 1回生成すると実測時間から secPerChar を更新し、同じページ内の次回以降は実測ベースで予測。\n")
    f.write("            const firstRunCorrection = secPerChar == null ? 12.35 : 1;\n")
    f.write("            return Math.max(isCpuMode ? 5 : 3, Math.round(rawEstimate * firstRunCorrection + pauseSeconds));\n")
    f.write("        }\n")
    f.write("        function updateEstimation() { const text = document.getElementById('ttsText').value || ''; const spokenText = text.replace(/\\[br(\\d+(?:\\.\\d+)?)\\]/gi, ''); const stepsEl = document.getElementById('numSteps'); const steps = stepsEl ? Number(stepsEl.value || 180) : 100; document.getElementById('charCounter').innerText = `目安: 約${Math.round(getEstimatedDuration())}秒 (${spokenText.length}文字・${steps}ステップ)`; }\n")
    f.write("        function filterCat(cat) { activeCategory = cat; ['all','male','female','neutral','other'].forEach(t=>{const btn=document.getElementById('tab-'+t); if(btn) btn.className='filter-tab';}); document.getElementById('tab-'+cat).className='filter-tab active'; renderPresets(); }\n")
    f.write("        function renderPresets() {\n")
    f.write("            const list = document.getElementById('presetList');\n")
    f.write("            list.innerHTML = presets.filter(p=>{\n")
    f.write("                if (activeCategory === 'all') return true;\n")
    f.write("                if (activeCategory === 'other') return (p.cat === 'unspecified' || p.cat === 'other');\n")
    f.write("                return p.cat === activeCategory;\n")
    f.write("            }).map(p=>{\n")
    f.write("                let b = 'badge-other';\n")
    f.write("                if (p.cat === 'male') b = 'badge-male';\n")
    f.write("                else if (p.cat === 'female') b = 'badge-female';\n")
    f.write("                else if (p.cat === 'neutral') b = 'badge-neutral';\n")
    f.write("                return `<div class=\"preset-card\" id=\"${p.id}\" onclick=\"applyPreset('${p.cat}', '${p.desc}', '${p.text}', '${p.id}')\"><div class=\"flex justify-between items-center w-full\"><span class=\"text-xs font-bold text-gray-800\">${p.title}</span><span class=\"badge ${b}\">${p.badge}</span></div><div class=\"text-[11px] text-gray-500 mt-1.5 overflow-hidden\" style=\"display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical;\">${p.desc}</div></div>`;\n")
    f.write("            }).join('');\n")
    f.write("        }\n")
    f.write("        function applyPreset(cat, desc, text, id) {\n")
    f.write("            document.getElementById('voiceDescription').value = desc;\n")
    f.write("            document.getElementById('ttsText').value = text;\n")
    f.write("            document.querySelectorAll('.preset-card').forEach(c=>c.classList.remove('active'));\n")
    f.write("            const activeEl = document.getElementById(id); if(activeEl) activeEl.classList.add('active'); updateEstimation();\n")
    f.write("        }\n")
    f.write("        function switchMode(m) {\n")
    f.write("            document.getElementById('modeInput').value = m;\n")
    f.write("            if (m === 'clone') {\n")
    f.write("                document.getElementById('tabClone').className = 'tab-btn flex-1 py-2.5 text-center text-sm font-bold text-gray-800 rounded-lg bg-white shadow-sm border border-gray-200';\n")
    f.write("                document.getElementById('tabDesign').className = 'tab-btn flex-1 py-2.5 text-center text-sm font-bold text-gray-400 rounded-lg';\n")
    f.write("                document.getElementById('cloneArea').classList.remove('hidden'); document.getElementById('designArea').classList.add('hidden');\n")
    f.write("            } else {\n")
    f.write("                document.getElementById('tabDesign').className = 'tab-btn flex-1 py-2.5 text-center text-sm font-bold text-gray-800 rounded-lg bg-white shadow-sm border border-gray-200';\n")
    f.write("                document.getElementById('tabClone').className = 'tab-btn flex-1 py-2.5 text-center text-sm font-bold text-gray-400 rounded-lg';\n")
    f.write("                document.getElementById('designArea').classList.remove('hidden'); document.getElementById('cloneArea').classList.add('hidden');\n")
    f.write("            }\n")
    f.write("        }\n")



    f.write("        let initElapsed = 0;\n")
    f.write("        let totalInitTime = 120; // GPUの目安。CPU判定後に240秒へ調整\n")
    f.write("        let initProgressValue = 8;\n")
    f.write("        const initTimerInterval = setInterval(() => {\n")
    f.write("            initElapsed += 1;\n")
    f.write("            const timerEl = document.getElementById('initTimerText');\n")
    f.write("            const ratio = Math.min(0.92, Math.pow(initElapsed / Math.max(1, totalInitTime), 0.85) * 0.92);\n")
    f.write("            const proposed = 8 + 84 * ratio; // 目安時間を過ぎても完了前は100%にしない\n")
    f.write("            initProgressValue = Math.max(initProgressValue, Math.min(92, proposed));\n")
    f.write("            if (timerEl) timerEl.innerText = `ロード中（経過時間：${initElapsed}秒／目安：約${totalInitTime}秒・進捗：約${Math.floor(initProgressValue)}%）`;\n")
    f.write("            const bar = document.getElementById('initProgressBar');\n")
    f.write("            if (bar) bar.style.width = initProgressValue + '%';\n")
    f.write("        }, 1000);\n")



    f.write("        async function checkStatus() {\n")
    f.write("            try {\n")
    f.write("                const res = await fetch('/status', { cache: 'no-store' });\n")
    f.write("                if (!res.ok) throw new Error('状態確認APIの応答エラー: HTTP ' + res.status);\n")
    f.write("                const data = await res.json();\n")
    f.write("                if (data.device === 'CPU') { isCpuMode = true; totalInitTime = 240; document.getElementById('cpuWarningBanner').classList.remove('hidden'); document.getElementById('headerDeviceBadge').innerHTML = '<span class=\"text-xs text-yellow-600 bg-yellow-100 px-2 py-1 rounded\">⚠️ CPUモード</span>'; }\n")
    f.write("                else { isCpuMode = false; totalInitTime = 120; document.getElementById('headerDeviceBadge').innerHTML = '<span class=\"text-xs text-green-600 bg-green-100 px-2 py-1 rounded\">⚡ GPU高速モード</span>'; }\n")
    f.write("                updateEstimation();\n")
    f.write("                if (data.status === 'ready') {\n")
    f.write("                    clearInterval(initTimerInterval);\n")
    f.write("                    document.getElementById('initTimerText').innerText = `読み込み完了（所要時間：${initElapsed}秒）`;\n")
    f.write("                    document.getElementById('initProgressBar').style.width = '100%';\n")
    f.write("                    const ov = document.getElementById('loadingOverlay');\n")
    f.write("                    ov.style.opacity = '0'; setTimeout(()=>ov.classList.add('hidden'), 1000);\n")
    f.write("                } else if (data.status === 'error') {\n")
    f.write("                    clearInterval(initTimerInterval);\n")
    f.write("                    const msg = data.message || 'モデルの初期化に失敗しました。';\n")
    f.write("                    document.getElementById('statusText').innerText = msg;\n")
    f.write("                    document.getElementById('statusText').classList.remove('text-gray-400'); document.getElementById('statusText').classList.add('text-red-400');\n")
    f.write("                    document.getElementById('initTimerText').innerText = '初期化に失敗しました。Colabの出力ログを確認してください。';\n")
    f.write("                    document.getElementById('initProgressBar').classList.remove('bg-yellow-500'); document.getElementById('initProgressBar').classList.add('bg-red-500');\n")
    f.write("                    document.getElementById('initProgressBar').style.width = '100%';\n")
    f.write("                } else {\n")
    f.write("                    document.getElementById('statusText').innerText = data.message || 'システムを準備中です...';\n")
    f.write("                    setTimeout(checkStatus, 1000);\n")
    f.write("                }\n")
    f.write("            } catch(e) { document.getElementById('statusText').innerText = '状態を確認できません。接続を再試行しています...'; setTimeout(checkStatus, 1500); }\n")
    f.write("        }\n")



    f.write("        let progressTimer = null; let elapsed = 0.0;\n")
    f.write("        function startProgressSimulation() {\n")
    f.write("            const bar = document.getElementById('progressBar');\n")
    f.write("            const statusTxt = document.getElementById('progressStatus');\n")
    f.write("            const timerTxt = document.getElementById('liveTimer');\n")
    f.write("            const detailTxt = document.getElementById('progressDetailText');\n")
    f.write("            elapsed = 0.0; const target = getEstimatedDuration(); lastEstimate = target;\n")
    f.write("            bar.style.width = '3%';\n")
    f.write("            timerTxt.innerText = `目安 約 ${target} 秒`;\n")
    f.write("            progressTimer = setInterval(() => {\n")
    f.write("                elapsed += 0.2;\n")
    f.write("                // 進捗は経過時間の目安。完了までは92%で止める。\n")
    f.write("                const percent = Math.min(92, 3 + (elapsed / Math.max(1, target)) * 75); const currentWidth = parseFloat(bar.style.width) || 0; bar.style.width = Math.max(currentWidth, percent) + '%';\n")
    f.write("                timerTxt.innerText = elapsed < target ? `経過 ${Math.floor(elapsed)} 秒 / 目安 約 ${target} 秒` : `予測時間を超過：処理を継続中（経過 ${Math.floor(elapsed)} 秒）`;\n")
    f.write("                detailTxt.innerText = `バーは経過時間から計算した目安です。実際の内部進捗は取得できないため、完了までは92%で止まります。`;\n")
    f.write("                statusTxt.innerText = elapsed < target ? '🔊 音声を合成しています...' : '⚡ 音声合成を継続しています...';\n")
    f.write("            }, 200);\n")
    f.write("        }\n")
    f.write("        function stopProgress(success) { clearInterval(progressTimer); if(success) document.getElementById('progressBar').style.width = '100%'; }\n")
    f.write("        async function pollTask(id) {\n")
    f.write("            try {\n")
    f.write("                const res = await fetch(`/task/${id}`); const data = await res.json();\n")
    f.write("                if (data.status === 'completed') {\n")
    f.write("                    stopProgress(true);\n")
    f.write("                    if (data.generation_time && data.generation_time > 0) {\n")
    f.write("                        const textLen = (document.getElementById('ttsText').value || '').length;\n")
    f.write("                        if (textLen > 0) {\n")
    f.write("                            const stepsEl = document.getElementById('numSteps');\n")
    f.write("                            const selectedSteps = stepsEl ? Number(stepsEl.value || 180) : 100;\n")
    f.write("                            // 実測値を100ステップ基準に換算し、次回以降の目安へ反映\n")
    f.write("                            const measuredRateAt100 = Math.max(0.008, (data.generation_time / textLen) * (100 / selectedSteps));\n")
    f.write("                            secPerChar = secPerChar == null ? measuredRateAt100 : (secPerChar * 0.15 + measuredRateAt100 * 0.85);\n")
    f.write("                        }\n")
    f.write("                    }\n")
    f.write("                    setTimeout(() => {\n")
    f.write("                        document.getElementById('progressArea').classList.add('hidden');\n")
    f.write("                        const player = document.getElementById('audioPlayer');\n")
    f.write("                        player.src = `/download/${id}`;\n")
    f.write("                        player.load();\n")
    f.write("                        player.preload = 'auto';\n")
    f.write("                        let autoplayScheduled = false;\n")
    f.write("                        player.oncanplay = () => {\n")
    f.write("                            if (autoplayScheduled) return;\n")
    f.write("                            autoplayScheduled = true;\n")
    f.write("                            player.oncanplay = null;\n")
    f.write("                            // 通信・デコードが落ち着くよう、再生可能になってから一呼吸（1000ms）待つ\n")
    f.write("                            setTimeout(() => { player.play().catch(() => { player.controls = true; }); }, 1000);\n")
    f.write("                        };\n")
    f.write("                        document.getElementById('downloadBtn').href = `/download/${id}`;\n")
    f.write("                        document.getElementById('resultArea').classList.remove('hidden');\n")
    f.write("                    }, 300);\n")
    f.write("                    document.getElementById('generateBtn').disabled = false;\n")
    f.write("                } else if (data.status === 'failed') {\n")
    f.write("                    alert('生成に失敗しました: ' + data.error); stopProgress(false);\n")
    f.write("                    document.getElementById('generateBtn').disabled = false;\n")
    f.write("                } else { setTimeout(()=>pollTask(id), 1000); }\n")
    f.write("            } catch(e) { setTimeout(()=>pollTask(id), 1500); }\n")
    f.write("        }\n")
    f.write("        document.getElementById('ttsForm').onsubmit = async (e) => {\n")
    f.write("            e.preventDefault();\n")
    f.write("            const btn = document.getElementById('generateBtn'); btn.disabled = true;\n")
    f.write("            document.getElementById('resultArea').classList.add('hidden');\n")
    f.write("            document.getElementById('progressArea').classList.remove('hidden');\n")
    f.write("            startProgressSimulation();\n")
    f.write("            const formData = new FormData(e.target);\n")
    f.write("            try {\n")
    f.write("                const res = await fetch('/generate', { method: 'POST', body: formData });\n")
    f.write("                const data = await res.json();\n")
    f.write("                if (!res.ok) { alert(data.error || 'エラーが発生しました'); stopProgress(false); btn.disabled = false; document.getElementById('progressArea').classList.add('hidden'); return; }\n")
    f.write("                pollTask(data.task_id);\n")
    f.write("            } catch(err) { alert('通信エラーが発生しました'); stopProgress(false); btn.disabled = false; document.getElementById('progressArea').classList.add('hidden'); }\n")
    f.write("        };\n")
    f.write("        checkStatus();\n")
    f.write("        renderPresets();\n")
    f.write("    </script>\n")
    f.write("</body>\n</html>\n")



print("環境をクリーンアップしています")



# 同じノートブック内で再実行された場合、前回起動したプロセスを先に終了する。
def _stop_process(proc, label):
    if proc is None:
        return
    try:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=3)
    except Exception:
        pass



_stop_process(globals().get("p"), "Cloudflare Tunnel")
_stop_process(globals().get("app_process"), "FastAPI")



# 変数が残っていないケース（セル中断後など）にも対応。対象ポートの占有プロセスだけを終了する。
try:
    subprocess.run(["fuser", "-k", "7861/tcp"], stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL, timeout=5, check=False)
except Exception:
    pass
# このノートブックの前回実行で残った cloudflared を終了する。
try:
    subprocess.run(["pkill", "-x", "cloudflared"], stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL, timeout=5, check=False)
except Exception:
    pass



# ポート解放がOSに反映されるのを少し待つ
for _ in range(10):
    check = subprocess.run(["fuser", "7861/tcp"], capture_output=True, text=True)
    if check.returncode != 0:
        break
    time.sleep(0.5)



print("アプリを起動しています")



# 6. サーバーを起動し、起動成功したかをポート監視でチェックする
log_path = "/content/server_error.log"
log_file = open(log_path, "w", encoding="utf-8")
app_process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=repo_dir,
    stdout=log_file,
    stderr=log_file
)



# サーバーが実際にHTTP応答するまで最大20秒待機。古いポート占有を誤認しない。
server_started = False
for _ in range(40):
    if app_process.poll() is not None:
        break
    try:
        import urllib.request
        with urllib.request.urlopen(f"http://127.0.0.1:7861/status?key={SECRET_KEY}", timeout=1) as response:
            if response.status < 500:
                server_started = True
                break
    except Exception:
        pass
    time.sleep(0.5)



if not server_started:
    print("アプリを起動できませんでした。起動ログを確認してください。")
    _stop_process(app_process, "FastAPI")
    try:
        log_file.flush()
        with open(log_path, "r", encoding="utf-8", errors="replace") as _lf:
            _log_lines = _lf.readlines()
        print("\n--- /content/server_error.log（末尾80行）---")
        print("".join(_log_lines[-80:]) or "ログは空です。")
    except Exception as _log_error:
        print(f"ログの読み込みに失敗しました: {_log_error}")
    log_file.close()
else:
    # Cloudflare URLの取得に上限時間を設け、URLが出ないままセルが固まるのを防ぐ。
    import select
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7861"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    tunnel_deadline = time.time() + 60
    tunnel_lines = []
    base_url = None
    while time.time() < tunnel_deadline and p.poll() is None:
        ready, _, _ = select.select([p.stdout], [], [], 0.5)
        if not ready:
            continue
        line = p.stdout.readline()
        if not line:
            continue
        tunnel_lines.append(line.rstrip())
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            base_url = match.group(0)
            break



    if base_url:
        secure_url = f"{base_url}/?key={SECRET_KEY}"
        print(f"アクセス用URL：\n{secure_url}")
    else:
        print("アクセス用URLを取得できませんでした。Tunnelのログを確認してください。")
        print("\n--- Cloudflare Tunnelログ（末尾40行）---")
        print("\n".join(tunnel_lines[-40:]) or "Tunnelからログを取得できませんでした。")
        _stop_process(p, "Cloudflare Tunnel")
        _stop_process(app_process, "FastAPI")
        log_file.close()



print(f"起動処理の経過時間：{(time.perf_counter() - _setup_started_at) / 60:.1f} 分")

AI音声スタジオをセットアップ
目安時間：約20〜60秒
環境をクリーンアップしています
アプリを起動しています
アクセス用URL：
https://compilation-appeared-subjects-announces.trycloudflare.com/?key=RQz5yj8r7l5aTf6Ee0P3Ea5gvpEvQ0TjMfFN6so7l0jOxVx813FJKuwGjRnZfxUrLLjWBOjq-RIApeCCIpAUUA
起動処理の経過時間：0.4 分




---



### 🟡サイトにアクセスできなくなったときは、下記手順で再実行
#### ノートブック上部 ＞ ランタイム ＞ [セッションを再起動してすべて実行する]ボタンを押下


### 🔴しばらく使わない場合は、下記手順で終了（無料枠のGPUは1日約4時間の上限）
#### ノートブック上部 ＞ ランタイム ＞ [ランタイムを接続解除して削除]ボタンを押下
※ブラウザやタブを落とした場合は、一定時間経過後に自動削除される
